# Day 16 — Simplified: Why Use MULTIPLE Attention "Heads"?

## The Question

Day 15's attention has ONE set of Q/K/V weights, so it learns ONE way to look at the sentence.

But a sentence has MANY kinds of relationships:

```
"The cat that I saw yesterday sat on the mat."

  Subject-verb:    cat ↔ sat        (HEAD 1 might learn this)
  Verb-object:     sat ↔ mat        (HEAD 2)
  Time word:       yesterday        (HEAD 3 — distant context)
  Adjacent words:  the-cat, sat-on  (HEAD 4)
```

One attention head can't handle ALL of these well. So we use MANY heads in parallel.

## The Committee of Specialists Analogy

```
One doctor (single-head attention):
  "I'll diagnose by looking at everything in general."
  → Generalist. Sometimes misses details.

A team of specialists (multi-head):
  "I'll listen to my heart specialist, eye specialist, etc."
  → Each focuses on one thing. Together they cover everything.
```

Multi-head attention = run several attention computations in parallel, then COMBINE their outputs.

## The Math (Simplest View)

```
Instead of one attention with embed_dim=128:

Run 4 attentions in parallel, each with head_dim=32:
   head_1 → (B, T, 32)
   head_2 → (B, T, 32)
   head_3 → (B, T, 32)
   head_4 → (B, T, 32)
                   ↓ concatenate
   combined → (B, T, 128)            ← back to original dim!
                   ↓ output projection W_O
   final     → (B, T, 128)
```

`head_dim = embed_dim / num_heads`. Same total compute, but split into specialists.

## What Different Heads Learn (Empirically)

When researchers visualize trained attention models, they see:

- Some heads attend to the **previous word** (like a bigram)
- Some attend to the **subject of the verb**
- Some attend to **matching brackets/quotes**
- Some attend to **the very first word** (sentence-level signal)
- Some heads are nearly useless (could be pruned)

We can't tell each head what to learn — they figure it out during training.

## Why It's Fast

Naively, multi-head looks slower (more heads = more work). But the trick is to reshape ONE big matrix into multiple "head slices":

```
W_Q is (embed_dim, embed_dim) — ONE big matrix
After Q = x @ W_Q:    Q has shape (B, T, embed_dim)
RESHAPE to:           (B, num_heads, T, head_dim)

Now all heads' attention runs in PARALLEL on the GPU.
```

Same parameter count as single-head attention. Same compute. But multiple specialized views.

## Real Numbers

| Model | num_heads | head_dim |
|-------|-----------|----------|
| GPT-2 small | 12 | 64 |
| GPT-3 | 96 | 128 |
| Llama-2 7B | 32 | 128 |

More heads = more potential specialization, but smaller head_dim per head. There's a sweet spot.

## TL;DR

```
Single-head: one perspective on the sentence
Multi-head:  many perspectives in parallel, then combined
```

Same compute. Better results. Standard in every transformer.

See `notebook.ipynb` for the full implementation with the reshape trick + speed comparison.